# Single Image Analysis - Refactored API

This notebook demonstrates the complete workflow using the refactored `colokroll` package structure.

**Steps:**
1. Load image and metadata
2. Background subtraction per channel (CUDA-accelerated)
3. Cell segmentation via Cellpose API
4. Colocalization analysis with mask filtering
5. Export results as DataFrames


In [ ]:
from pathlib import Path
import time
import logging

import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
import pandas as pd

# Refactored imports
from colokroll import (
    ImageLoader,
    ImageIOConfig,
    BackgroundSubtractor,
)
from colokroll.analysis.segmentation import segment_cells
from colokroll.analysis.colocalization import (
    compute_colocalization,
    export_colocalization_json,
    estimate_min_area_threshold,
)

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')


## 1. Load Image and Rename Channels

In [ ]:
# Update this path to your image file
image_path = Path("/path/to/your/image.ome.tiff")

image_loader = ImageLoader(ImageIOConfig(verbose=True))
loaded_data = image_loader.load_image(image_path)
pixel_size = image_loader.get_pixel_size()

print(f"Loaded image shape: {loaded_data.shape}")
print(f"Pixel size: {pixel_size} μm")
print(f"Original channel names: {image_loader.get_channel_names()}")

# Rename channels to match your experiment
new_channel_names = ['DAPI', 'ALIX', 'Phalloidin', 'LAMP1']
image_loader.rename_channels(new_channel_names)
channel_names = image_loader.get_channel_names()
print(f"Renamed channels: {channel_names}")


## 2. Background Subtraction (CUDA-accelerated)

The `BackgroundSubtractor` automatically searches for the best method and parameters.


In [ ]:
bg_subtractor = BackgroundSubtractor()

results = {}

for i, ch in enumerate(channel_names):
    ch_data = loaded_data[:, :, :, i]
    t0 = time.perf_counter()
    
    corrected, meta = bg_subtractor.subtract_background(
        image=ch_data,
        channel_name=ch,
        # method omitted -> auto search + full run
    )
    
    cp.cuda.Stream.null.synchronize()
    dt = time.perf_counter() - t0
    
    results[ch] = (corrected, meta)
    print(f"\n{ch}: {dt:.2f}s -> {meta['method']}")

### Visualize Background Subtraction Results


In [ ]:
middle_slice_idx = loaded_data.shape[0] // 2

fig = bg_subtractor.plot_background_subtraction_comparison(
    original_data=loaded_data,
    corrected_results=results,
    channel_names=channel_names,
    z_slice=middle_slice_idx,
    figsize=(5 * len(channel_names), 12)
)
plt.show()


## 3. Cell Segmentation via Cellpose API

Segment cells using the simplified `segment_cells()` function. This automatically creates a weighted composite from Phalloidin and DAPI MIPs and handles all the API interactions with built-in retry logic.

**Key improvement**: Uses background-subtracted data for better segmentation results!


In [ ]:
# Extract background-subtracted channels and create MIPs
# Using the corrected data for better segmentation results
ph_corrected = results["Phalloidin"][0]  # Get corrected array from (array, metadata) tuple
da_corrected = results["DAPI"][0]

# Create MIPs from background-subtracted data
ph_mip = ph_corrected.max(axis=0).astype(np.float32)
da_mip = da_corrected.max(axis=0).astype(np.float32)

# Segment using the simplified interface - this replaces 50+ lines of code!
save_dir = Path("./outputs/cellpose")
dst_mask, dst_outl, mask = segment_cells(
    phalloidin_mip=ph_mip,
    dapi_mip=da_mip,
    output_dir=save_dir,
    filename_stem=f"{image_path.stem}_phall_dapi",
    phalloidin_weight=0.8,
    dapi_weight=0.2,
    resize_values=(600, 400),  # Try 600 first, fallback to 400
    pause_seconds=1.0,
)

print(f"✓ Segmentation complete!")
print(f"  Found {mask.max()} cells")
print(f"  Mask saved to: {dst_mask}")
print(f"  Outlines saved to: {dst_outl}")

# Visualize mask
plt.figure(figsize=(6, 6))
plt.title(f"Segmentation Mask ({mask.max()} cells)")
plt.imshow(mask, cmap="tab20")
plt.axis("off")
plt.show()


## 4. Colocalization Analysis

Compute colocalization between two channels (e.g., ALIX and LAMP1) using the segmentation mask.


In [ ]:
# Load the mask we just created
mask_path = dst_mask

# Estimate minimum area threshold
min_area = estimate_min_area_threshold(mask_path, fraction_of_median=0.70)

# Run colocalization
res = compute_colocalization(
    image=results,  # dict[str, (array, meta)] or dict[str, array]
    mask=mask_path,
    channel_a="ALIX",
    channel_b="LAMP1",
    thresholding='costes',
    max_border_fraction=0.10,
    min_area=int(min_area),
    border_margin_px=1,
    plot_mask=True,
)

print("\nTotal image metrics:")
print(res["results"]["total_image"])


## 5. Export Results as DataFrames

In [ ]:
# Per-cell dataframe (one row per kept label)
df_cells = pd.DataFrame(res["results"]["per_label"]).sort_values("label")

# Total-image (single row)
df_total = pd.DataFrame([res["results"]["total_image"]])

# Optional: save to CSV
# df_cells.to_csv("per_cell_metrics.csv", index=False)
# df_total.to_csv("total_image_metrics.csv", index=False)

print("Per-cell metrics:")
df_cells

In [ ]:
print("Total image metrics:")
df_total

## Summary

This notebook demonstrated:
- Loading images with `colokroll.io.ImageLoader`
- CUDA-accelerated background subtraction with auto parameter selection
- **Simplified cell segmentation** with `segment_cells()` - just one function call!
- Colocalization analysis with mask filtering
- Exporting results as DataFrames

### Key Improvements:
- **Simpler segmentation**: Reduced from 50+ lines to a single function call
- **Better results**: Uses background-subtracted data for segmentation
- **More robust**: Built-in retry logic and error handling
- **Cleaner code**: No manual composite creation or API management needed

All functionality uses the refactored package structure:
- `colokroll.io` for image loading
- `colokroll.preprocessing` for background subtraction
- `colokroll.analysis.segmentation` for segmentation (now with simplified interface!)
- `colokroll.analysis.colocalization` for colocalization metrics
